# Classifier SpaCy
Utilisation de textcat dans la pipeline nlp spacy pour tester la classification à partir de sample de 512 tokens de chaque histoire

In [1]:
import pandas as pd

In [3]:
df_cp = pd.read_csv('cp.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'cp.csv'

In [4]:
df = df_cp[df_cp.source.isin(['reddit','fandom'])]
df['texte'] = df.texte.apply(lambda x : x[:1000000])
df_fandom = df[df.source == 'fandom'].sample(2000)
df_reddit= df[df.source == 'reddit'].sample(2000)

df_train = pd.concat([df_fandom,df_reddit])
cats= df_train.source.unique().tolist()

/var/folders/4m/_y65_yfx68b53909_pvhthr40000gn/T/ipykernel_10098/1623927441.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['texte'] = df.texte.apply(lambda x : x[:1000000])


In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.tokens import Doc, Span
from collections import Counter
from typing import List, Tuple, Set

def make_docs_chunked(
    nlp,
    data: List[Tuple[str, str]],
    target_file: str,
    cats: Set[str],
    max_tokens: int = 512,
    overlap: int = 50
):
    """
    Convertit des textes longs en chunks sans perdre les embeddings,
    gr�ce � span.as_doc(). Ajoute les labels pour textcat.
    """
    docs = DocBin(store_user_data=True)
    class_counter = Counter()

    for text, label in data:
        # DOC ORIGINAL AVEC EMBEDDINGS
        doc = nlp(text)
        n_tokens = len(doc)

        start = 0
        while start < n_tokens:
            end = min(start + max_tokens, n_tokens)

            # Le chunk : Span \u2192 Doc (garde les vecteurs !)
            chunk_span = doc[start:end]
            chunk_doc = chunk_span.as_doc()

            # Ajouter les labels (binary exclusive)
            for cat in cats:
                chunk_doc.cats[cat] = 1 if cat == label else 0

            docs.add(chunk_doc)
            class_counter[label] += 1

            if end == n_tokens: 
                break

            start = end - overlap  # overlap token-based

    docs.to_disk(target_file)

    print(f"\u2714 DocBin saved: {target_file}")
    print(f"\U0001f4ca Distribution des chunks par classe : {dict(class_counter)}")

    return docs


In [ ]:
data = list(zip(df_train['texte'],df_train['source']))

In [ ]:
import os
import shutil
import spacy
from spacy.tokens import DocBin
from spacy.cli.train import train as spacy_train
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
import numpy as np
from tqdm import tqdm
import pandas as pd
import logging

# ======================================================
# LOGGING GLOBAL
# ======================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s \u2014 %(levelname)s \u2014 %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger(__name__)
logger.info("\U0001f680 Démarrage de la pipeline de cross-validation spaCy + chunking")


# ======================================================
# 1. Pr�paration des donn�es : chunking
# ======================================================

logger.info("\U0001f4cc Étape 1 : Préparation des données & chunking")

# Exemple : data = [(text, label), ...]
# \U0001f449 � adapter � ton dataset r�el
# data = list(zip(df["texte"], df["label"]))

logger.info(f"Taille du dataset brut : {len(data)} exemples")

cats = set(label for _, label in data)
logger.info(f"Classes détectées : {cats}")

# Mod�le Spacy minimal
nlp_raw = spacy.load('en_core_web_sm')
logger.info("Modèle tokenizer chargé : en_core_web_sm")

# Construction du fichier ALL.SPACY
logger.info("\u23f3 Génération des chunks\u2026")

make_docs_chunked(
    nlp=nlp_raw,
    data=data,
    target_file="spacy_textcat/all.spacy",
    cats=cats,
    max_tokens=512,
    overlap=50
)

logger.info("\u2714 Chunking termin�")

# ======================================================
# 2. Charger les chunks
# ======================================================

logger.info("\U0001f4cc Étape 2 : Chargement des chunks")

alldocs = DocBin().from_disk("spacy_textcat/all.spacy")
docs = list(alldocs.get_docs(nlp_raw.vocab))

logger.info(f"Nombre total de chunks chargés : {len(docs)}")


# ======================================================
# 3. R�cup�rer les labels
# ======================================================

logger.info("\U0001f4cc Étape 3 : Extraction des labels pour la stratification")

nlp_tmp = spacy.load("output_spacy/spacy_textcat_update/model-best")
labels = list(nlp_tmp.get_pipe("textcat").labels)

logger.info(f"Labels dans le modèle : {labels}")

y_all = []
for doc in docs:
    y_all.append(labels[np.argmax([doc.cats[l] for l in labels])])

logger.info(f"Répartition des labels (chunks) : {pd.Series(y_all).value_counts().to_dict()}")


# ======================================================
# 4. Cross-validation stratifi�e
# ======================================================

logger.info("\U0001f4cc Étape 4 : Cross-validation stratifiée")

k = 5
kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
reports = []

for fold, (train_idx, dev_idx) in enumerate(kf.split(docs, y_all)):

    logger.info(f"\n===== \U0001f501 FOLD {fold+1}/{k} =====")

    fold_dir = f"cv/fold_{fold}"
    os.makedirs(fold_dir, exist_ok=True)

    logger.info(f"\U0001f4c2 Dossier créé : {fold_dir}")
    logger.info(f"Taille train : {len(train_idx)} \u2014 Taille dev : {len(dev_idx)}")

    # -------------------------
    # TRAIN/DEV .spacy
    # -------------------------
    db_train = DocBin()
    db_dev = DocBin()

    for i in train_idx:
        db_train.add(docs[i])
    for i in dev_idx:
        db_dev.add(docs[i])

    db_train.to_disk(f"{fold_dir}/train.spacy")
    db_dev.to_disk(f"{fold_dir}/dev.spacy")

    logger.info("\U0001f4be train.spacy & dev.spacy écrits")

    # -------------------------
    # Copier config.cfg
    # -------------------------
    shutil.copy("/home/alexandrelionnet/Documents/phd/script/spacy_textcat/config_cpu.cfg", f"{fold_dir}/config.cfg")
    logger.info("\U0001f4c4 config.cfg copié dans le dossier du fold")

    # -------------------------
    # Entra�nement via spacy_train
    # -------------------------
    logger.info("\u23f3 Entraînement du modèle\u2026")

    spacy_train(
        f"{fold_dir}/config.cfg",
        output_path=f"{fold_dir}/model",
        overrides={
            "paths.train": f"{fold_dir}/train.spacy",
            "paths.dev": f"{fold_dir}/dev.spacy",
        },
    )

    logger.info("\u2714 Entraînement terminé pour ce fold")

    # -------------------------
    # �VALUATION
    # -------------------------
    logger.info("\u23f3 Évaluation du modèle\u2026")

    nlp_fold = spacy.load(f"{fold_dir}/model/model-best")

    y_true = []
    y_pred = []

    for doc in tqdm([docs[i] for i in dev_idx], desc=f"Fold {fold} eval"):
        gold = doc.cats
        pred = nlp_fold(doc.text).cats

        true_label = labels[np.argmax([gold[l] for l in labels])]
        pred_label = labels[np.argmax([pred[l] for l in labels])]

        y_true.append(true_label)
        y_pred.append(pred_label)

    report = classification_report(
        y_true, y_pred, target_names=labels, output_dict=True, zero_division=0
    )
    reports.append(report)

    logger.info(f"\U0001f4ca Rapport pour le fold {fold+1} ajout�")

# ======================================================
# 5. Moyenne finale des folds
# ======================================================

logger.info("\U0001f4cc Étape 5 : Moyenne finale sur les folds")

results_df = pd.DataFrame(reports)
mean_results = results_df.mean(numeric_only=True)

logger.info("\n===== MOYENNE DES FOLDS =====")
logger.info(mean_results.to_string())

print("\n===== MOYENNE DES FOLDS =====")
print(mean_results)


In [ ]:
# Aplatir les colonnes contenant des dictionnaires
df_flat = results_df.applymap(
    lambda d: d['precision'] if isinstance(d, dict) else d
).add_prefix('precision_')

df_flat['recall_fandom'] = results_df['fandom'].apply(lambda d: d['recall'])
df_flat['f1_fandom'] = results_df['fandom'].apply(lambda d: d['f1-score'])

df_flat['recall_reddit'] = results_df['reddit'].apply(lambda d: d['recall'])
df_flat['f1_reddit'] = results_df['reddit'].apply(lambda d: d['f1-score'])

df_flat['precision_macro'] = results_df['macro avg'].apply(lambda d: d['precision'])
df_flat['recall_macro']    = results_df['macro avg'].apply(lambda d: d['recall'])
df_flat['f1_macro']        = results_df['macro avg'].apply(lambda d: d['f1-score'])

df_flat['precision_weighted'] = results_df['weighted avg'].apply(lambda d: d['precision'])
df_flat['recall_weighted']    = results_df['weighted avg'].apply(lambda d: d['recall'])
df_flat['f1_weighted']        = results_df['weighted avg'].apply(lambda d: d['f1-score'])

df_flat


# Régression logisitque : une autre approche 
En réutilisant la méthode utilisée pour le cas canon / non_canon, Application d'un logit sur les features formelles et thématique.

In [29]:
import pandas as pd 
df = pd.read_csv('/Users/rolly/Documents/10-19_Université_et_scolarité/PhD/script/output/data_pronoms.csv')
df = df[df.source.isin(['fandom','reddit'])]

## if reddit == 0 , if fandom ==1
df.source= df.source.apply(lambda x : 1 if x=='fandom' else 0)

num_features=['longueur_texte', 'nombre_mots',
       'longueur_phrase', 'Flesch Reading Ease', 'Flesch-Kincaid Grade Level',
       'Gunning Fog Index', 'SMOG Index', 'Automated Readability Index',
       'Coleman-Liau Index', 'Dale-Chall Readability Score', 'Indice Lix',
       'Ratio Types/Tokens', 'Hapax Legomena', 'Densité Lexicale',
       "Indice Honore's R", 'fear', 'neutral', 'disgust', 'anger', 'sadness',
       'surprise', 'joy', 'noun_verb_ratio', 'noun_adj_verb_ratio',
       'passive_verb_ratio', 'nll','Ratio_P1_rest']
df= df[num_features + ['source']]

/var/folders/4m/_y65_yfx68b53909_pvhthr40000gn/T/ipykernel_9581/228725182.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/rolly/Documents/10-19_Université_et_scolarité/PhD/script/output/data_pronoms.csv')


In [31]:
# ==============================================
# PIPELINE LOGIT ROBUSTE + MÉTRIQUES CLASSIQUES
# ==============================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("=== 1. Nettoyage des données ===")

df_clean = df.copy()

# Filtrage textes trop courts
df_clean = df_clean[df_clean["nombre_mots"] > 30]

# Gestion inf / NaN
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna()

print("Shape après nettoyage :", df_clean.shape)

# --------------------------------------
# Winsorisation 1%-99%
# --------------------------------------

def winsorize_series(s, lower=0.01, upper=0.99):
    return s.clip(s.quantile(lower), s.quantile(upper))

num_cols = df_clean.select_dtypes(include=np.number).columns
num_cols = num_cols.drop("source")

for col in num_cols:
    df_clean[col] = winsorize_series(df_clean[col])

print("Winsorisation effectuée.")

# --------------------------------------
# Log-transform variables heavy-tailed
# --------------------------------------

heavy_cols = [
    "longueur_texte",
    "nombre_mots",
    "Hapax Legomena",
    "Indice Honore's R",
    "Indice Lix"
]

for col in heavy_cols:
    if col in df_clean.columns:
        df_clean[f"log_{col}"] = np.log1p(df_clean[col])

print("Log-transform effectuée.")

# --------------------------------------
# Séparation X / y
# --------------------------------------

y = df_clean["source"]
X = df_clean.drop(columns=["source"])

print("\n=== 2. Sélection L1 avec validation croisée ===")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegressionCV(
        penalty="l1",
        solver="saga",
        cv=cv,
        scoring="roc_auc",
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    ))
])

pipeline.fit(X, y)

# Performance AUC
y_pred = pipeline.predict_proba(X)[:, 1]
auc = roc_auc_score(y, y_pred)

print("AUC (cross-validated) :", round(auc, 4))

# Variables sélectionnées
coefs = pd.Series(
    pipeline.named_steps["model"].coef_[0],
    index=X.columns
)

selected_vars = coefs[coefs != 0].index.tolist()

print("\nVariables sélectionnées :", len(selected_vars))
print(selected_vars)

# --------------------------------------
# 3. Refit Logit classique statsmodels
# --------------------------------------

print("\n=== 3. Refit Logit classique (statsmodels) ===")

X_selected = X[selected_vars]
X_selected = sm.add_constant(X_selected)

logit_model = sm.Logit(y, X_selected)
result = logit_model.fit(disp=0)

print(result.summary())

# --------------------------------------
# 4. Métriques classiques
# --------------------------------------

print("\n=== 4. Métriques classiques ===")

llf = result.llf          # log-likelihood modèle
llnull = result.llnull    # log-likelihood modèle nul
llr = result.llr          # likelihood ratio test
llr_pvalue = result.llr_pvalue
pseudo_r2 = result.prsquared
aic = result.aic
bic = result.bic

print(f"Log-Likelihood (model)      : {llf:.3f}")
print(f"Log-Likelihood (null model) : {llnull:.3f}")
print(f"LLR statistic               : {llr:.3f}")
print(f"LLR p-value                 : {llr_pvalue:.5f}")
print(f"Pseudo R² (McFadden)        : {pseudo_r2:.4f}")
print(f"AIC                         : {aic:.2f}")
print(f"BIC                         : {bic:.2f}")

# --------------------------------------
# 5. Odds ratios
# --------------------------------------

print("\n=== 5. Odds Ratios ===")

odds_ratios = np.exp(result.params)
conf = result.conf_int()
conf_odds = np.exp(conf)

odds_table = pd.DataFrame({
    "Odds Ratio": odds_ratios,
    "CI Lower": conf_odds[0],
    "CI Upper": conf_odds[1],
    "p-value": result.pvalues
})

print(odds_table.sort_values("Odds Ratio", ascending=False))

=== 1. Nettoyage des données ===
Shape après nettoyage : (24868, 28)
Winsorisation effectuée.
Log-transform effectuée.

=== 2. Sélection L1 avec validation croisée ===
AUC (cross-validated) : 0.7968

Variables sélectionnées : 32
['longueur_texte', 'nombre_mots', 'longueur_phrase', 'Flesch Reading Ease', 'Flesch-Kincaid Grade Level', 'Gunning Fog Index', 'SMOG Index', 'Automated Readability Index', 'Coleman-Liau Index', 'Dale-Chall Readability Score', 'Indice Lix', 'Ratio Types/Tokens', 'Hapax Legomena', 'Densité Lexicale', "Indice Honore's R", 'fear', 'neutral', 'disgust', 'anger', 'sadness', 'surprise', 'joy', 'noun_verb_ratio', 'noun_adj_verb_ratio', 'passive_verb_ratio', 'nll', 'Ratio_P1_rest', 'log_longueur_texte', 'log_nombre_mots', 'log_Hapax Legomena', "log_Indice Honore's R", 'log_Indice Lix']

=== 3. Refit Logit classique (statsmodels) ===
                           Logit Regression Results                           
Dep. Variable:                 source   No. Observations:   

In [ ]:
##pinguin

import numpy as np
import pingouin as pg
import pandas as pd 

df = pd.read_csv("/Users/rolly/Documents/10-19_Université_et_scolarité/PhD/script/output/data_pronoms.csv")



/var/folders/4m/_y65_yfx68b53909_pvhthr40000gn/T/ipykernel_9581/2936433463.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/rolly/Documents/10-19_Université_et_scolarité/PhD/script/output/data_pronoms.csv")


In [3]:
df_res = pd.DataFrame()
metrics = ['longueur_texte', 'nombre_mots',
       'longueur_phrase', 'Flesch Reading Ease', 'Flesch-Kincaid Grade Level',
       'Gunning Fog Index', 'SMOG Index', 'Automated Readability Index',
       'Coleman-Liau Index', 'Dale-Chall Readability Score', 'Indice Lix',
       'Ratio Types/Tokens', 'Hapax Legomena', 'Densité Lexicale',
       "Indice Honore's R", 'fear', 'neutral', 'disgust', 'anger', 'sadness',
       'surprise', 'joy', 'noun_verb_ratio', 'noun_adj_verb_ratio',
       'passive_verb_ratio','Ratio_P1_rest']
for metric in metrics :
    x = df[df.source == 'fandom'][metric].dropna()
    y= df[df.source == 'reddit'][metric].dropna()
    

    res = pg.ttest(x,y)
    res['metrics']= metric
    df_res = pd.concat([df_res,res])

df_res.set_index('metrics')


/var/folders/4m/_y65_yfx68b53909_pvhthr40000gn/T/ipykernel_9581/2454407455.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_res = pd.concat([df_res,res])
/var/folders/4m/_y65_yfx68b53909_pvhthr40000gn/T/ipykernel_9581/2454407455.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_res = pd.concat([df_res,res])
/var/folders/4m/_y65_yfx68b53909_pvhthr40000gn/T/ipykernel_9581/2454407455.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecate

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
metrics,,,,,,,,
longueur_texte,10.957803,12344.300623,two-sided,8.187318e-28,"[2488.72, 3573.07]",0.150892,1.419e+24,1.000000
nombre_mots,10.427667,12372.566187,two-sided,2.361111e-25,"[417.38, 610.62]",0.143557,5.045e+21,1.000000
longueur_phrase,0.042949,24187.646718,two-sided,9.657422e-01,"[-0.24, 0.25]",0.000546,0.014,0.050211
Flesch Reading Ease,-10.013286,24755.984938,two-sided,1.477469e-23,"[-2.02, -1.36]",0.126271,7.452e+19,NaN
Flesch-Kincaid Grade Level,3.250042,24932.829174,two-sided,1.155408e-03,"[0.07, 0.3]",0.040498,2.806,0.890201
Gunning Fog Index,2.773435,24916.446470,two-sided,5.550908e-03,"[0.05, 0.27]",0.034538,0.669,0.775922
SMOG Index,12.936889,19784.207758,two-sided,3.979230e-38,"[0.2, 0.27]",0.169487,2.271e+34,NaN
Automated Readability Index,5.317945,24953.961450,two-sided,1.058486e-07,"[0.24, 0.53]",0.066331,1.951e+04,0.999445
Coleman-Liau Index,28.611400,20059.696408,two-sided,1.680695e-176,"[0.57, 0.65]",0.374228,9.337e+172,1.000000


In [32]:
df_res

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power,metrics
T-test,10.957803,12344.300623,two-sided,8.187318e-28,"[2488.72, 3573.07]",0.150892,1.419e+24,1.000000,longueur_texte
T-test,10.427667,12372.566187,two-sided,2.361111e-25,"[417.38, 610.62]",0.143557,5.045e+21,1.000000,nombre_mots
T-test,0.042949,24187.646718,two-sided,9.657422e-01,"[-0.24, 0.25]",0.000546,0.014,0.050211,longueur_phrase
T-test,-10.013286,24755.984938,two-sided,1.477469e-23,"[-2.02, -1.36]",0.126271,7.452e+19,NaN,Flesch Reading Ease
T-test,3.250042,24932.829174,two-sided,1.155408e-03,"[0.07, 0.3]",0.040498,2.806,0.890201,Flesch-Kincaid Grade Level
T-test,2.773435,24916.446470,two-sided,5.550908e-03,"[0.05, 0.27]",0.034538,0.669,0.775922,Gunning Fog Index
T-test,12.936889,19784.207758,two-sided,3.979230e-38,"[0.2, 0.27]",0.169487,2.271e+34,NaN,SMOG Index
T-test,5.317945,24953.961450,two-sided,1.058486e-07,"[0.24, 0.53]",0.066331,1.951e+04,0.999445,Automated Readability Index
T-test,28.611400,20059.696408,two-sided,1.680695e-176,"[0.57, 0.65]",0.374228,9.337e+172,1.000000,Coleman-Liau Index
T-test,15.762134,19918.292979,two-sided,1.229846e-55,"[0.17, 0.22]",0.206336,6.421e+51,1.000000,Dale-Chall Readability Score
